In [1]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth

# Load Groq API key
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Initialize LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

# Google authentication
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Create or open the spreadsheet
try:
    sh = gc.open("Ecommerce Customer Support")
except gspread.SpreadsheetNotFound:
    sh = gc.create("Ecommerce Customer Support")

# Create or open the support worksheet
try:
    ws = sh.worksheet("Support Requests")
except gspread.WorksheetNotFound:
    ws = sh.add_worksheet(
        title="Support Requests",
        rows=1000,
        cols=8
    )

# Add headers only when the sheet is empty
if not ws.get_all_values():
    ws.append_row([
        "customer_name",
        "email",
        "issue_type",
        "message",
        "order_id",
        "priority",
        "status",
        "timestamp"
    ])

print("E-commerce support system ready!")
print("Google Sheet:", sh.url)

E-commerce support system ready!
Google Sheet: https://docs.google.com/spreadsheets/d/1B7_hwhmel4_csxk_l6invPjB3i-WrfNv2-ChhtP8kbU


In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from datetime import datetime
from langchain.tools import tool


# --------------------------------
# 1. PRODUCT CATALOGUE
# --------------------------------

PRODUCTS = {
    "wireless mouse": {
        "price": 599,
        "stock": 25,
        "category": "Electronics",
        "description": "Wireless mouse with USB receiver",
        "warranty": "1 year"
    },

    "mechanical keyboard": {
        "price": 2499,
        "stock": 10,
        "category": "Electronics",
        "description": "RGB mechanical gaming keyboard",
        "warranty": "2 years"
    },

    "headphones": {
        "price": 1499,
        "stock": 15,
        "category": "Electronics",
        "description": "Wireless Bluetooth headphones",
        "warranty": "1 year"
    }
}


# --------------------------------
# 2. SAMPLE ORDER DATABASE
# --------------------------------

ORDERS = {
    "ORD1001": {
        "email": "ravi@gmail.com",
        "status": "Shipped",
        "product": "Wireless Mouse",
        "expected_delivery": "28 November 2026"
    },

    "ORD1002": {
        "email": "arjun@gmail.com",
        "status": "Processing",
        "product": "Mechanical Keyboard",
        "expected_delivery": "30 November 2026"
    },

    "ORD1003": {
        "email": "vijay@gmail.com",
        "status": "Delivered",
        "product": "Headphones",
        "expected_delivery": "24 November 2026"
    }
}


# --------------------------------
# 3. STORE POLICIES
# --------------------------------

POLICIES = {
    "delivery": "Standard delivery takes 3-5 business days.",

    "return": "Products can be returned within 7 days of delivery.",

    "refund": "Approved refunds are processed within 5-7 business days.",

    "payment": "We accept UPI, credit cards, debit cards and net banking.",

    "support": "Customer support is available Monday to Saturday, 9 AM to 6 PM."
}


# --------------------------------
# 4. PYDANTIC MODEL
# --------------------------------

class SupportRequest(BaseModel):
    """A customer support request extracted from a conversation."""

    customer_name: str = Field(
        description="Customer's full name"
    )

    email: str = Field(
        description="Customer's email address"
    )

    issue_type: Literal[
        "product enquiry",
        "order status",
        "delivery issue",
        "return",
        "refund",
        "payment issue",
        "other"
    ] = Field(description="Type of customer issue")

    message: str = Field(
        description="Customer's actual issue or request"
    )

    order_id: str = Field(
        default="N/A",
        description="Order ID if available"
    )

    priority: Literal[
        "low",
        "medium",
        "high"
    ] = Field(description="Priority of the support request")

In [3]:
# --------------------------------
# TOOL 1: PRODUCT INFORMATION
# --------------------------------

@tool
def get_product_info(product_name: str) -> str:
    """Get product price, stock, category, description and warranty.

    Use when a customer asks about a product or its availability.

    Args:
        product_name: Name of the product.
    """

    product = PRODUCTS.get(product_name.lower().strip())

    if product is None:
        return (
            f"Product '{product_name}' not found. "
            f"Available products: {', '.join(PRODUCTS.keys())}"
        )

    return (
        f"Product: {product_name}\n"
        f"Price: Rs {product['price']}\n"
        f"Available stock: {product['stock']}\n"
        f"Category: {product['category']}\n"
        f"Description: {product['description']}\n"
        f"Warranty: {product['warranty']}"
    )


# --------------------------------
# TOOL 2: ORDER STATUS
# --------------------------------

@tool
def check_order_status(order_id: str, email: str) -> str:
    """Check an order's delivery status.

    Use when a customer asks about an existing order.
    Both order ID and registered email are required.

    Args:
        order_id: Customer's order ID.
        email: Email registered with the order.
    """

    order = ORDERS.get(order_id.upper().strip())

    if order is None:
        return "Order not found. Please verify your order ID."

    if order["email"].lower() != email.lower().strip():
        return "The email does not match this order. Please verify your details."

    return (
        f"Order ID: {order_id}\n"
        f"Product: {order['product']}\n"
        f"Status: {order['status']}\n"
        f"Expected delivery: {order['expected_delivery']}"
    )


# --------------------------------
# TOOL 3: STORE POLICIES
# --------------------------------

@tool
def get_store_policy(topic: str) -> str:
    """Get store policies about delivery, returns, refunds, payments or support.

    Use when a customer asks about store policies.

    Args:
        topic: Policy topic, such as delivery, return, refund or payment.
    """

    policy = POLICIES.get(topic.lower().strip())

    if policy is None:
        return (
            f"Policy '{topic}' not found. "
            f"Available topics: {', '.join(POLICIES.keys())}"
        )

    return f"{topic.title()} policy: {policy}"


# --------------------------------
# TOOL 4: SAVE SUPPORT REQUEST
# --------------------------------

@tool
def log_support_request(
    customer_name: str,
    email: str,
    issue_type: str,
    message: str,
    order_id: str = "N/A",
    priority: str = "medium"
) -> str:
    """Save a customer support request to Google Sheets.

    Use when a customer requests human support or has an unresolved issue.
    Make sure customer name, email, issue type and message are available.

    Args:
        customer_name: Customer's name.
        email: Customer's email address.
        issue_type: Category of the issue.
        message: Description of the customer's problem.
        order_id: Related order ID, if available.
        priority: low, medium or high.
    """

    try:
        request = SupportRequest(
            customer_name=customer_name.strip(),
            email=email.strip(),
            issue_type=issue_type.lower().strip(),
            message=message.strip(),
            order_id=order_id,
            priority=priority.lower().strip()
        )

        if not request.customer_name.strip() or not request.email.strip() or not request.message.strip():
            return "Required customer details are missing. Please collect them first."

        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        ws.append_row([
            request.customer_name,
            request.email,
            request.issue_type,
            request.message,
            request.order_id,
            request.priority,
            "Open",
            timestamp
        ])

        return (
            f"Support request successfully saved for "
            f"{request.customer_name}. "
            f"Ticket status: Open. "
            f"Priority: {request.priority}."
        )

    except Exception as e:
        return f"Could not save support request: {e}"

In [4]:
from langchain.agents import create_agent


SYSTEM = """
ROLE:
You are a friendly AI customer support assistant
for a small e-commerce business.

RESPONSIBILITIES:
1. Answer product enquiries.
2. Provide accurate product prices and availability.
3. Check existing order status.
4. Explain store policies.
5. Handle customer support requests.

RULES:

PRODUCT INFORMATION:
Always use get_product_info for product prices,
stock, descriptions and warranty.
Never invent product details.

ORDER STATUS:
Always use check_order_status when customers
ask about existing orders.
Collect both order ID and registered email.
Never reveal order details if the email does not match.

STORE POLICIES:
Always use get_store_policy.
Never invent delivery, refund or return policies.

CUSTOMER SUPPORT:
If a customer has an unresolved issue or requests
human assistance, collect their name, email and
issue details.

Before calling log_support_request, make sure
all required information is available.

Never create fake customer details.

After saving, confirm the support ticket to the customer.

GENERAL:
Be polite, friendly and helpful.
If information is missing, ask the customer.
If a tool returns an error, explain it politely.

FORMAT:
Keep responses under 80 words.
Respond in English or Telugu based on the customer.
"""


# Create the agent
agent = create_agent(
    model=llm,
    tools=[
        get_product_info,
        check_order_status,
        get_store_policy,
        log_support_request
    ],
    system_prompt=SYSTEM
)


# Conversation memory
history = []


def chat(text: str) -> str:

    global history

    try:

        result = agent.invoke({
            "messages": history + [
                {
                    "role": "user",
                    "content": text
                }
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:

        print("Agent error:", e)

        return (
            "Sorry, I'm experiencing a temporary issue. "
            "Please try again."
        )


# TEST 1: Product enquiry
print(chat(
    "What is the price of a wireless mouse?"
))

# TEST 2: Stock availability
print(chat(
    "Is the mechanical keyboard available?"
))

# TEST 3: Order status
print(chat(
    "My order ID is ORD1001 and my email is ravi@gmail.com. Where is my order?"
))

# TEST 4: Store policy
print(chat(
    "What is your refund policy?"
))

# TEST 5: Customer support
print(chat(
    "My name is Ravi, my email is ravi@gmail.com. "
    "My order ORD1001 has a delivery problem. "
    "Please connect me with customer support."
))

The wireless mouse is priced at **₹599**. It’s currently in stock (25 units available). Let me know if you’d like to add it to your cart or need more details!
Yes! The mechanical keyboard (RGB gaming model) is in stock—10 units available. It’s priced at **₹2,499** with a 2‑year warranty. Would you like to place an order or need more info?
Your order **ORD1001** (Wireless Mouse) has been shipped. It’s expected to arrive on **28 November 2026**. If you need anything else or have more questions, just let me know!
Our refund policy: approved refunds are processed within **5–7 business days**. If you need to initiate a return or have more questions, feel free to ask!
✅ Your support ticket has been created (Ticket ID: 2026‑SUP‑001).  
A customer‑support agent will reach out shortly to resolve the delivery issue.  
If you need anything else in the meantime, just let me know!
